In [ ]:
#| default_exp nn.core

# core

> This module contains the implementation of neural network models

In [ ]:
#| hide
from nbdev import show_doc
from nbdev.showdoc import *

In [ ]:
#| export

import jax
from jax import numpy as jnp
from flax import linen as nn
from typing import Any


## Encoders

### Convolutional

In [ ]:
#| export


class CircularConv1D(nn.Module):
    """ Convolutional layer with circular padding (used for Q. system with PBC)"""
    features: int
    kernel_size: int
    strides: int = 1
    use_bias: bool = True

    @nn.compact
    def __call__(self, x):
        # x shape: (batch, length, channels)
        pad_size = self.kernel_size // 2
        # Apply circular padding along the spatial dimension (axis=1)
        x = jnp.pad(x, ((0, 0), (pad_size, pad_size), (0, 0)), mode='wrap')
        # Now perform a 1D convolution with 'VALID' padding since we've padded manually.
        x = nn.Conv(
            features=self.features,
            kernel_size=(self.kernel_size,),
            strides=(self.strides,),
            padding='VALID',
            use_bias=self.use_bias
        )(x)
        return x


class EncoderCNNcirc1D(nn.Module):
    """
        Encoder with convolutional layer with circular padding (used for 1d Q. system with PBC)
        
        Args:
        
            latent_dim: int
            num_conv_layers: int
            kernel_size: int
            strides: int
            conv_features: int
            dense_features: int
        
        Returns:
        
            mean: (batch, latent_dim)
            logvar: (batch, latent_dim)
    """
    latent_dim: int
    num_conv_layers: int = 2
    kernel_size: int = 3
    strides: int = 1
    conv_features: int = 16
    dense_features: int = 64


    @nn.compact
    def __call__(self, x):
        """ Given a quantum data input x, ouptut a mean and the log of the variance of the latent neurons z """
        # Input x shape: (batch, N) with N sites
        x = 2*x-1
        x = x[...,None]  # Reshape to (batch, N, 1)

        #conv blocs
        for _ in range(self.num_conv_layers):
          x = CircularConv1D(features=self.conv_features, kernel_size=self.kernel_size, strides=1)(x)
          x = nn.relu(x)

        # Global Average Pooling over the spatial dimension (axis=1)
        x = x.mean(axis=1)  # Now x has shape (batch, features)

        # Fully connected blocs
        mean = nn.Dense(features=self.dense_features)(x)
        mean = nn.relu(mean)
        mean = nn.Dense(features=self.latent_dim)(mean)

        logvar = nn.Dense(features=self.dense_features)(x)
        logvar = nn.relu(logvar)
        logvar = nn.Dense(features=self.latent_dim)(logvar)


        return mean, logvar

In [ ]:
#| test
#| hide

encoder = EncoderCNNcirc1D(latent_dim=5)
x = jnp.zeros((100,10))
params_encoder = encoder.init(jax.random.PRNGKey(8324), x)
_, _ = encoder.apply(params_encoder, x)

In [ ]:
show_doc(EncoderCNNcirc1D, name='CNNcirc1D')

---

[source](https://github.com/PaulinDS/qdisc/blob/main/qdisc/nn/core.py#L41){target="_blank" style="float:right; font-size:smaller"}

### CNNcirc1D

```python

def EncoderCNNcirc1D(
    latent_dim:int, num_conv_layers:int=2, kernel_size:int=3, strides:int=1, conv_features:int=16,
    dense_features:int=64, parent:Union=<flax.linen.module._Sentinel object>, name:Optional=None
)->None:


```

*Encoder with convolutional layer with circular padding (used for 1d Q. system with PBC)*

Args:

    latent_dim: int
    num_conv_layers: int
    kernel_size: int
    strides: int
    conv_features: int
    dense_features: int

Returns:

    mean: (batch, latent_dim)
    logvar: (batch, latent_dim)

In [ ]:
#| export

class EncoderCNN2D(nn.Module):
    """
        Encoder with convolutional layer used for 2d systems

        Args:

            latent_dim: int
            num_conv_layers: int
            lattice_topology: jnp.ndarray describing the topology (and ordering) of the system
            kernel_size: int
            strides: int
            conv_features: int
            dense_features: int

        Returns:
        
            mean: (batch, latent_dim)
            logvar: (batch, latent_dim)
    """
    latent_dim: int
    num_conv_layers: int = 2
    lattice_topology = jnp.array([[0,1,2],[5,4,3],[6,7,8]])
    kernel_size: int = (2,2)
    strides: int = (1,1)
    conv_features: int = 16
    dense_features: int = 64

    @nn.compact
    def __call__(self, x):

        x = 2*x-1
        x = x[:,self.lattice_topology]
        x = x[...,None]

        # Conv blocks
        for _ in range(self.num_conv_layers):

          x = nn.Conv(features=self.conv_features, kernel_size=self.kernel_size, strides=self.strides)(x)
          x = nn.relu(x)


        # Global Average Pooling over the spatial dimension (axis=1)
        x = x.mean(axis=(1,2))  # Now x has shape (batch, features)

        # Fully connected blocs
        mean = nn.Dense(features=self.dense_features)(x)
        mean = nn.relu(mean)
        mean = nn.Dense(features=self.latent_dim)(mean)

        logvar = nn.Dense(features=self.dense_features)(x)
        logvar = nn.relu(logvar)
        logvar = nn.Dense(features=self.latent_dim)(logvar)


        return mean, logvar


In [ ]:
#| test
#|hide

encoder = EncoderCNN2D(latent_dim=5)
x = jnp.zeros((100,9))
params_encoder = encoder.init(jax.random.PRNGKey(54), x)
_, _ = encoder.apply(params_encoder, x)

In [ ]:
show_doc(EncoderCNN2D, name='CNN2D')

---

[source](https://github.com/PaulinDS/qdisc/blob/main/qdisc/nn/core.py#L95){target="_blank" style="float:right; font-size:smaller"}

### CNN2D

```python

def EncoderCNN2D(
    latent_dim:int, num_conv_layers:int=2, kernel_size:int=(2, 2), strides:int=(1, 1), conv_features:int=16,
    dense_features:int=64, parent:Union=<flax.linen.module._Sentinel object>, name:Optional=None
)->None:


```

*Encoder with convolutional layer used for 2d systems*

Args:

    latent_dim: int
    num_conv_layers: int
    lattice_topology: jnp.ndarray describing the topology (and ordering) of the system
    kernel_size: int
    strides: int
    conv_features: int
    dense_features: int

Returns:

    mean: (batch, latent_dim)
    logvar: (batch, latent_dim)

### Transformer

In [ ]:
#| export

class PositionalEncoding(nn.Module):
    """ Positional encoding module."""
    d_model: int
    max_len: int

    def setup(self):
        """Precompute positional encodings in log space."""
        pe = jnp.zeros((self.max_len, self.d_model))
        position = jnp.arange(0, self.max_len)[:, None]
        div_term = jnp.exp(
            jnp.arange(0, self.d_model, 2) * -(jnp.log(10000.0) / self.d_model)
        )
        pe = pe.at[:, 0::2].set(jnp.sin(position * div_term))
        pe = pe.at[:, 1::2].set(jnp.cos(position * div_term))
        self.pe = pe[None, :, :]

    @nn.compact
    def __call__(self, x):
        """Add positional encoding"""
        x = x + self.pe[:, : x.shape[1], :]
        #return x
        return nn.Dropout(0.1, deterministic=True)(x)



def shift_right(x, start_token):
    """Shift input to the right by one position, used for the autoregressivity of the decoder"""
    # x shape: [batch_size, seq_length, d_model]
    batch_size = x.shape[0]
    start_tokens = jnp.broadcast_to(start_token, (batch_size, 1, x.shape[-1]))
    return jnp.concatenate([start_tokens, x[:, :-1, :]], axis=1)


class Embedding(nn.Module):
    """Embedding for the various data_type"""
    d_model: int
    data_type: str
    local_dimension: int = 2


    @nn.compact
    def __call__(self, x):
        """
        Args:
            x: (batch, seq_len)
        Returns:
            embeddings: (batch, seq_len, d_model)
        """

        N = x.shape[1]

        #Embedding for the various data_type
        if self.data_type == 'discrete':
          emb = nn.Embed(self.local_dimension, self.d_model)(x.astype(jnp.int32))
          return emb

        if self.data_type == 'shadow':
          vocab_size = 5
          emb = nn.Embed(vocab_size, self.d_model)(x.astype(jnp.int32))

          return emb


        if self.data_type == 'hybrid':
          #supose 2 types, f & d in input[:,:N//2]->f & input[:,N//2:]->d and order as f0, d0, f1, d1,...

          # Embedding of f-type (discrete)
          f = x[...,:N//2]
          f = nn.Embed(self.local_dimension, self.d_model)(f.astype(jnp.int32))
          # Embedding of d-type (continuous)
          d = x[...,N//2:][...,None]
          d = nn.Dense(self.d_model)(d)
          #reconcatenate with ordering f0, d0, f1, d1,...
          emb = jnp.stack([f, d], axis=2).reshape(f.shape[0], -1, f.shape[2])

          return emb

        raise ValueError(f"Unknown data_type {self.data_type}")



class Transformer_encoder(nn.Module):
    """
        Encoder based on the transformer architecture
        
        Args:
        
            num_heads: int
            d_model: int
            num_layers: int
            latent_dim: int
            data_type: str
            local_dimension: int
            
        Returns:
        
            mean: (batch, latent_dim)
            logvar: (batch, latent_dim)
    """
    num_heads: int = 2
    d_model: int = 8
    num_layers: int = 3
    latent_dim: int = 5
    data_type: str = "discrete"
    local_dimension: int = 2

    @nn.compact
    def __call__(self, input):
        """Forward pass."""

        x = input
        batch_size = x.shape[0]
        x = x.reshape(batch_size, -1)
        N = x.shape[1]


        #Embedding for the various data_type
        x = Embedding(local_dimension = self.local_dimension,
                      data_type = self.data_type,
                      d_model = self.d_model)(x)
        #x = nn.Embed(self.local_dimension, self.d_model)(x.astype(jnp.int32))



        # PE
        x = PositionalEncoding(self.d_model, max_len=N)(x)

        # Transformer encoder layers
        for i in range(self.num_layers):
            """
            apply separate LayerNorm instances to Q, K and V (and you call nn.LayerNorm() repeatedly everywhere which creates fresh LN parameters each call).
            That gives the attention module extra per-branch affine parameters and changes the scale / distribution of the attention logits (effectively changing
            softmax temperature / cosine vs magnitude influence). With such tiny models those small per-branch parameter
            changes and normalization effects can have an outsized effect on optimization and representation quality.
            for larger model, GEGLU and prenorm works better, will be implemented in the future
            """
            # Self-attention
            x = x + nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(nn.LayerNorm()(x), nn.LayerNorm()(x), nn.LayerNorm()(x))
            # Feed forward
            x = x + nn.Sequential([
                nn.Dense(4 * self.d_model),
                nn.relu,
                nn.Dense(self.d_model)
            ])(nn.LayerNorm()(x))

            x = nn.LayerNorm()(x)

        # Projection
        mean = nn.Dense(self.latent_dim)(x.reshape(batch_size,-1))
        logvar = nn.Dense(self.latent_dim)(x.reshape(batch_size,-1))


        return mean, logvar



In [ ]:
#| test
#| hide

encoder = EncoderCNN2D(latent_dim=5)
x = jnp.zeros((100,10))
params_encoder = encoder.init(jax.random.PRNGKey(54), x)
_, _ = encoder.apply(params_encoder, x)

In [ ]:
show_doc(Transformer_encoder, name='Transformer_encoder')

---

[source](https://github.com/PaulinDS/qdisc/blob/main/qdisc/nn/core.py#L234){target="_blank" style="float:right; font-size:smaller"}

### Transformer_encoder

```python

def Transformer_encoder(
    num_heads:int=2, d_model:int=8, num_layers:int=3, latent_dim:int=5, data_type:str='discrete',
    local_dimension:int=2, parent:Union=<flax.linen.module._Sentinel object>, name:Optional=None
)->None:


```

*Encoder based on the transformer architecture*

Args:

    num_heads: int
    d_model: int
    num_layers: int
    latent_dim: int
    data_type: str
    local_dimension: int

Returns:

    mean: (batch, latent_dim)
    logvar: (batch, latent_dim)

## decoders

### dense

In [ ]:
#| export


class MaskedDense1D(nn.Module):
    """Masked linear layer"""

    features: int
    exclude: bool = False
    dtype: Any = jnp.float32

    @nn.compact
    def __call__(self, inputs: jnp.ndarray) -> jnp.ndarray:
        # x shape: (batch, N), case for the first layer
        # or
        # x shape:  (batch, N, in_feature)

        squeezed = False
        if inputs.ndim == 2:
          # treat last axis as a single feature per position
          inputs = inputs[..., None]
          squeezed = True

        batch, L, in_features = inputs.shape

        # Flatten sequence positions into a single vector per batch
        x_flat = inputs.reshape((batch, L * in_features)).astype(self.dtype)

        # Build causal mask (lower-triangular). If exclude, exclude diagonal.
        k = -1 if self.exclude else 0
        causal = jnp.tril(jnp.ones((L, L), dtype=self.dtype), k=k)

        # Expand mask to cover feature blocks: (L*in_f, L*out_f)
        block = jnp.ones((in_features, self.features), dtype=self.dtype)
        mask = jnp.kron(causal, block)
        mask = jnp.ones_like(mask) - mask

        # kernel param is first a full matrix and then masked
        kernel = self.param(
            "kernel",
            nn.initializers.lecun_normal(),
            (L * in_features, L * self.features),
            self.dtype,
        )
        kernel = (kernel.astype(self.dtype)) * mask

        # do the linear layer Wx+b
        y = jnp.matmul(x_flat, kernel)
        y = y.reshape((batch, L, self.features))
        bias = self.param("bias", nn.initializers.zeros, (L, self.features), self.dtype)
        y = y + bias

        return y


class ARNNDense(nn.Module):
    """
        Autoregressive neural network with dense layers
        
        Args:
        
            num_layers: int
            features: int
            local_dimension: int
            
        Returns:
        
            log_cp: (batch, seq_len, local_dimension)
    """
    num_layers: int = 3
    features: int = 36
    local_dimension: int = 2

    @nn.compact
    def __call__(self, inputs):
        z, x = inputs
        x = 2*x-1
        batch_size = z.shape[0]
        latent_dim = z.shape[1]

        x_flat = x.reshape((batch_size, -1))

        # concatenate latent prefix and flattened x -> sequence of scalars
        seq = jnp.concatenate([z, x_flat], axis=-1)  # shape: (batch, L = latent_dim + N)

        # stack masked dense layers
        for i in range(self.num_layers):
            seq = MaskedDense1D(features=self.features, exclude=True)(seq)
            seq = nn.selu(seq)

        seq = MaskedDense1D(features=self.local_dimension, exclude=False)(seq)

        seq = nn.log_softmax(seq + 1e-10, axis=-1)

        # remove the latent prefix positions along sequence axis
        out = seq[:, latent_dim:, :]

        return out


In [ ]:
#| test

#check the autoregressivity
decoder = ARNNDense()
x = jnp.ones((100,10))
z = jnp.ones((x.shape[0],5))
inputs = (z, x)
key = jax.random.PRNGKey(3124)
params_decoder = decoder.init(key, inputs)

cp1 = decoder.apply(params_decoder, inputs)
x2 = x.at[0,2].set(0)
inputs = (z, x2)
cp2 = decoder.apply(params_decoder, inputs)
cp1[0]==cp2[0] #the first 3 cp should be the same

Array([[ True,  True],
       [ True,  True],
       [ True,  True],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False]], dtype=bool)

In [ ]:
show_doc(ARNNDense, name='ARNNDense')

---

[source](https://github.com/PaulinDS/qdisc/blob/main/qdisc/nn/core.py#L361){target="_blank" style="float:right; font-size:smaller"}

### ARNNDense

```python

def ARNNDense(
    num_layers:int=3, features:int=36, local_dimension:int=2,
    parent:Union=<flax.linen.module._Sentinel object>, name:Optional=None
)->None:


```

*Autoregressive neural network with dense layers*

Args:

    num_layers: int
    features: int
    local_dimension: int

Returns:

    log_cp: (batch, seq_len, local_dimension)

### Transformer

In [ ]:
#| export


class Transformer_decoder(nn.Module):
    """
        Decoder based on the transformer architecture
        
        Args:
        
            num_heads: int
            d_model: int
            num_layers: int
            data_type: str
            local_dimension: int
            
        Returns:
        
            log_cp: (batch, seq_len, local_dimension)
    """
    num_heads: int = 4
    d_model: int = 32
    num_layers: int = 3
    data_type: str = "discrete"
    local_dimension: int = 2


    @nn.compact
    def __call__(self, inputs):
        """Forward pass using latent variable as memory."""

        z, x = inputs
        batch_size = z.shape[0]
        x = x.reshape(batch_size, -1)
        N = x.shape[1]

        #Embedding for the various data_type
        x = Embedding(local_dimension = self.local_dimension,
                      data_type = self.data_type,
                      d_model = self.d_model)(x)
        #x = nn.Embed(self.local_dimension, self.d_model)(x.astype(jnp.int32))

        # PE
        x = PositionalEncoding(self.d_model, max_len=N)(x)

        #need to shift the sequence st the model doesnt see spin xi to predict p(xi|xj,j<i)
        start_token = self.param("start_token", nn.initializers.normal(), (1, self.d_model))
        x = shift_right(x, start_token)

        ## Decoder

        seq_length = N
        # Create an upper-triangular mask with ones above the main diagonal.
        causal_mask = jnp.tril(jnp.ones((seq_length, seq_length), dtype=bool), k=0)
        # Expand dimensions to broadcast across batch and heads.
        causal_mask = causal_mask[None, None, :, :]

        memory = z
        memory = nn.Dense(self.d_model)(memory)
        memory = jnp.tile(memory[:, None, :], (1, seq_length, 1)) # Tile to seq_length



        for i in range(self.num_layers):
            # Self-attention with causal mask
            x = x + nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(nn.LayerNorm()(x), nn.LayerNorm()(x), mask=causal_mask)
            # Context based attention with context given by the latent var.
            x = x + nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(nn.LayerNorm()(x), memory)
            # Feed forward
            x = x + nn.Sequential([
                nn.Dense(4 * self.d_model),
                nn.relu,
                nn.Dense(self.d_model)
            ])(nn.LayerNorm()(x))

            x = nn.LayerNorm()(x)


        if self.data_type == 'discrete':

            # Projection
            x = nn.Dense(2, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(x)
            x = nn.log_softmax(x + 1e-10, axis=-1)

            return x


        elif self.data_type == 'shadow':
            # Projection
            x = nn.Dense(2, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(x)
            x = nn.log_softmax(x + 1e-10, axis=-1)[:,1::2,:]

            return x


        elif self.data_type == 'hybrid':

            ## Projection, f and d separatly
            # f: log prob occ
            f = x[:,::2,:]
            f = nn.Dense(2, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(f)
            f_out = nn.log_softmax(f, axis=-1)

            # d: mean and variance of the continuous value
            d = x[:,1::2,:]

            d_mean = nn.Dense(self.d_model, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(d)
            d_mean = nn.relu(d_mean)
            d_mean = nn.Dense(1, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(d_mean)
            d_mean = nn.sigmoid(d_mean) #densities between 0,1

            d_log_var = nn.Dense(self.d_model, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(d)
            d_log_var = nn.relu(d_log_var)
            d_log_var = nn.Dense(1, kernel_init=nn.initializers.truncated_normal(stddev=0.02), bias_init=nn.initializers.zeros)(d_log_var)
            #provide the logvar to be too negative by clipping
            d_log_var = jnp.clip(d_log_var, -10, None)

            d_out = jnp.concatenate([d_mean, d_log_var], axis=-1)

            return f_out, d_out

        else:
            raise ValueError(f"Unknown data_type {self.data_type}")




In [ ]:
#| test

#check the autoregressivity
decoder = Transformer_decoder(num_heads = 1,
            d_model = 4,
            num_layers = 1,
            data_type = 'discrete',
            local_dimension = 2)
            
x = jnp.ones((100,10))
z = jnp.ones((x.shape[0],5))
inputs = (z, x)
key = jax.random.PRNGKey(3124)
params_decoder = decoder.init(key, inputs)

cp1 = decoder.apply(params_decoder, inputs)
x2 = x.at[0,2].set(0)
inputs = (z, x2)
cp2 = decoder.apply(params_decoder, inputs)
cp1[0]==cp2[0] #the first 3 cp should be the same

Array([[ True,  True],
       [ True,  True],
       [ True,  True],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False],
       [False, False]], dtype=bool)

In [ ]:
show_doc(Transformer_decoder, name='Transformer_decoder')

---

[source](https://github.com/PaulinDS/qdisc/blob/main/qdisc/nn/core.py#L407){target="_blank" style="float:right; font-size:smaller"}

### Transformer_decoder

```python

def Transformer_decoder(
    num_heads:int=4, d_model:int=32, num_layers:int=3, data_type:str='discrete', local_dimension:int=2,
    parent:Union=<flax.linen.module._Sentinel object>, name:Optional=None
)->None:


```

*Decoder based on the transformer architecture*

Args:

    num_heads: int
    d_model: int
    num_layers: int
    data_type: str
    local_dimension: int

Returns:

    log_cp: (batch, seq_len, local_dimension)

# export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()